# RFD4-Proteina

![RFD4-Proteina](https://proto-bio.github.io/proto-assets/images/tool/rfd4/hero.png)

RFD4-Proteina is an all-atom flow-matching generative model that **co-generates sequence and
all-atom structure in a single pass**, so every design arrives with its residue identities and
packed sidechains already assigned and needs no separate inverse-folding step. A design task is
written as a `contig` topology plus a set of typed conditions — motif coordinates, interface
hotspots, burial, secondary structure, CATH fold class, per-identity sequence probabilities —
that steer one shared checkpoint. This notebook demonstrates both registered tools:
`run_rfd4_design` for joint sequence/structure generation, and `run_rfd4_inverse_folding` for
sequence and sidechain design on a backbone that is held fixed.


In [1]:
import tempfile
from pathlib import Path

from proto_tools.entities.structures import Structure
from proto_tools.tools.structure_design.rfd4 import (
    RFD4DesignConfig,
    RFD4DesignSpec,
    RFD4Input,
    RFD4InverseFoldingConfig,
    run_rfd4_design,
    run_rfd4_inverse_folding,
)
from proto_tools.tools.structure_design.rfd4.rfd4_design import EXAMPLE_INPUT_FIXTURE
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools, display_doc_link

# Resolved against the installed toolkit package, so this works in editable and installed layouts
INPUTS_DIR = EXAMPLE_INPUT_FIXTURE.parent / "examples" / "inputs"
PDL1_TARGET_PATH = INPUTS_DIR / "pdl1_target_5o45_cropped.cif"
MONOMER_SENTINEL_PATH = INPUTS_DIR / "unconditional_monomer_100.cif"

display_doc_link("rfd4")
display_available_tools("rfd4")

- **`run_rfd4_design()`** — Joint sequence and all-atom structure design with RFD4-Proteina
- **`run_rfd4_inverse_folding()`** — All-atom sequence and sidechain design on a fixed backbone with RFD4-Proteina

## Example inputs

Three inputs drive the cells below, and the two taken from upstream are the inputs upstream's own
documentation uses, so each run here reproduces an upstream worked example.

| Input | What it is | Used for |
|---|---|---|
| `example_input_fixture.cif` | 65-residue single-chain monomer that ships with the toolkit | motif scaffolding, inverse folding |
| `pdl1_target_5o45_cropped.cif` | PD-L1 (PDB 5O45) chain A, residues 17-131 — the target in upstream's `condition_spec_ppi.yaml` and `binder-in-10-lines.md` | binder design |
| `unconditional_monomer_100.cif` | upstream's unconditional benchmark input: one collapsed expandable-segment sentinel annotated `100-100` | unconditional generation |


In [2]:
# 65-residue monomer shipped with the toolkit
fixture = Structure.from_file(EXAMPLE_INPUT_FIXTURE)
print(f"Fixture: chains {fixture.get_chain_ids()}, {fixture.num_residues} residues")
print(f"  {fixture.get_chain_sequence('A')}")

# Upstream's binder-design target
pdl1_target = Structure.from_file(PDL1_TARGET_PATH)
print(f"PD-L1 target: chains {pdl1_target.get_chain_ids()}, {pdl1_target.num_residues} residues")
print(f"  {pdl1_target.get_chain_sequence('A')}")

# Upstream's unconditional benchmark input; a sentinel, so there is no real sequence to print
monomer_sentinel = Structure.from_file(MONOMER_SENTINEL_PATH)
print(f"Monomer sentinel: {MONOMER_SENTINEL_PATH.name} ({monomer_sentinel.structure_format})")

Fixture: chains ['A'], 65 residues
  MRKKLDLKKFVEDKNQEYAARALGLSQKLIEEVLKRGLPVYVETNKDGNIKVYITQDGITQPFPP
PD-L1 target: chains ['A'], 115 residues
  NAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVN
Monomer sentinel: unconditional_monomer_100.cif (cif)


## API reference

The tables below are generated from the Pydantic models and docstrings in
`rfd4_design.py`, so they always match the installed version. `rfd4-inverse-folding` shares the
same input and output models and a subset of the config, so only `rfd4-design` is shown here.


In [3]:
# Display input docs
display_api_reference("rfd4-design", "input", "run_rfd4_design")

**Input** — `RFD4Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>design_specs</code> | <code>list[RFD4DesignSpec]</code> | <code>[]</code> | Independent design tasks, each with its own input structure and constraints |
| <code>raw_json</code> | <code>str &#124; None</code> | <code>None</code> | Raw upstream condition-spec JSON, forwarded as-is (ignores design_specs) |

In [4]:
# Display config docs
display_api_reference("rfd4-design", "config", "run_rfd4_design")

**Config** — `RFD4DesignConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on; RFD4 requires a GPU |
| <code>num_replicates</code> | <code>int</code> | <code>1</code> | Designs generated per input spec |
| <code>batch_size</code> | <code>int</code> | <code>1</code> | Designs denoised in parallel on the GPU; confirm the longest input fits at 1 first |
| <code>nsteps</code> | <code>int</code> | <code>60</code> | Integration steps per design; must stay 60 unless recycle_strategy is None |
| <code>recycle_strategy</code> | <code>Optional[Literal['low/flat', 'low/front_loaded', 'low/midpeak', 'medium/flat', 'medium/front_loaded', 'medium/midpeak', 'high/flat', 'high/front_loaded', 'high/midpeak', 'thinking_low', 'thinking_medium', 'thinking_high']]</code> | <code>'thinking_medium'</code> | Trunk recycling schedule, e.g. 'medium/front_loaded'; pins nsteps to 60. None = no recycling |
| <code>backbone_sampling</code> | <code>RFD4SamplerTuning</code> | <code>RFD4SamplerTuning(tsr_k=None, tsr_sigma=None, noise_scale=None)</code> | Sampling temperature for the backbone modality; see RFD4SamplerTuning |
| <code>sequence_sampling</code> | <code>RFD4SamplerTuning</code> | <code>RFD4SamplerTuning(tsr_k=None, tsr_sigma=None, noise_scale=None)</code> | Sampling temperature for the sidechain/sequence latent modality |
| <code>deterministic_sampling</code> | <code>bool</code> | <code>False</code> | Integrate both modalities as an ODE, with no mid-trajectory noise injection |
| <code>model_checkpoint</code> | <code>str</code> | <code>'nvidia/RFD4-Proteina'</code> | HuggingFace repo id or a local model-bundle directory |
| <code>model_revision</code> | <code>str &#124; None</code> | <code>'a06e2509c7033b010fc43614e9fd840303588502'</code> | HuggingFace commit to pin for reproducible loads; ignored for local bundles |
| <code>adapter_path</code> | <code>str &#124; None</code> | <code>None</code> | Local PEFT/LoRA adapter directory to load onto the base model |
| <code>use_torch_compile</code> | <code>bool</code> | <code>False</code> | Compile the trunk across upstream length buckets; pays a long first-call compile |
| <code>save_trajectory_every</code> | <code>int</code> | <code>0</code> | Snapshot the trajectory every N steps (0 disables); requires output_dir to be set |
| <code>output_dir</code> | <code>str &#124; None</code> | <code>None</code> | Directory for run outputs, one fresh run-<id> subdir per call; None uses a tempdir |
| <code>guidance_w</code> | <code>float</code> | <code>1.0</code> | Classifier-free guidance weight; 1.0 = no guidance, 0.0 excludes the main model |
| <code>partial_trajectory_fraction_steps</code> | <code>float &#124; None</code> | <code>None</code> | Run only the final fraction of the trajectory; 0.3 stays close to the input |
| <code>partial_t_backbone</code> | <code>float &#124; None</code> | <code>None</code> | Backbone starting corruption; near 1 stays close to the input, near 0 regenerates |
| <code>partial_t_latent</code> | <code>float &#124; None</code> | <code>None</code> | Sequence/sidechain starting corruption; near 1 stays close to the input |
| <code>couple_partial_t</code> | <code>bool</code> | <code>False</code> | Derive the unset modality's corruption level from the one that is set |

In [5]:
# Display output docs
display_api_reference("rfd4-design", "output", "run_rfd4_design")

**Output** — `RFD4Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>designed_structures</code> | <code>list[RFD4Designs]</code> | <code>[]</code> | Per-spec bundles of generated designs |

**`RFD4Designs`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>spec_key</code> | <code>str</code> | required | Identifier of the input spec that produced these designs |
| <code>designs</code> | <code>list[RFD4Design]</code> | <code>[]</code> | Designs generated for this spec |

**`RFD4Design`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>structure</code> | <code>Structure</code> | required | The generated all-atom structure |
| <code>complex</code> | <code>Complex</code> | required | Predictor-ready chains and per-chain sequences |

## Writing a design spec

Each `RFD4DesignSpec` is one design task: an input structure, an optional `contig` describing the
topology to build, and typed condition fields describing what the model is told about it.

### Contig tokens

A contig is a single comma-separated string, read left to right:

| Token | Meaning |
|---|---|
| `A17-131` | preserved segment — residues 17 to 131 of chain `A` of the input structure |
| `A52` | preserved single residue |
| `20-30` | unnamed expandable segment; the model picks a length between 20 and 30 |
| `E1:2-20` | named expandable segment; the name becomes the segment's `res_name`, so conditions can address it as `sel('*/E1')` |
| `/0` | chain break — everything after it belongs to a new chain |

A trailing `P`, `R`, or `D` sets an expandable segment's polymer type (protein, RNA, DNA), so
`10R` is a ten-nucleotide RNA segment. Omitting `contig` keeps the whole input structure. Every
contig must contain at least one preserved segment: RFD4 always consumes an input structure, and
an all-expandable contig that never references it is rejected upstream.

> **The `contig` token forms match RFdiffusion3's, but the semantics do not.** `/0`, `A42-48` and
> `20-30` are written exactly as they are for `rfdiffusion3-design`; how they resolve is what
> differs. RFD4 resolves residue ids against `label_seq_id` rather than author numbering, and it
> requires at least one preserved segment because there is no `length` field to size a design on
> its own. The spec dialect that used to surround the contig is gone as well: `length`, `symmetry`,
> `unindex` and `select_*` are all rejected. RFdiffusion1's slash-separated contigs are a
> genuinely unrelated grammar. Re-read a contig against the tool you are about to run it on rather
> than copying one across tools.

### Selections

Every `select`-style field takes an atomworks query, either as a path —
`sel('<chain>/<res_name>/<res_id>/<atom_name>')`, with trailing fields defaulting to `*` — or as
a boolean query over annotations (`is_ligand`, `is_protein_backbone`, `has_nan_coord()`,
`chain_id == 'A'`), and the two can be combined with `and`. Path fields are positional, so
`sel('B/*/N')` reads `N` as a residue id; write `sel('B/*/*/N')` to reach an atom name.

### Residue numbering

Residue ids in contigs and selections are read as **author numbering** by default — the numbering
PDB files display and the rest of proto-tools uses — and the implementation translates them to the mmCIF
`label_seq_id` that RFD4 actually resolves against. Pass `residue_numbering="label"` to skip the
translation when a spec is already written in `label_seq_id` numbering, as it would be for an
annotated CIF round-tripped out of a previous RFD4 run. Both files used below happen to number
their chains identically in the two schemes — the fixture 1-65, the cropped PD-L1 target 17-131 —
so upstream's spec transcribes with its residue ids untouched and the translation is invisible
here. It is not invisible for a crystal structure whose label numbering restarts at 1 or whose
author numbering carries gaps.

### Integration steps and trunk recycling

Every `recycle_strategy` preset ships an `n_recycle_schedule` whose bands sum to exactly **60**
integration steps, and upstream asserts that the expansion equals `nsteps`. `RFD4CommonConfig`
enforces that up front rather than after the model has loaded: with a preset set — the default is
`thinking_medium` — `nsteps` must stay `60`, and a custom `nsteps` is legal only with
`recycle_strategy=None`.

What the presets change is how many trunk passes those 60 steps cost: `low/flat` is the cheapest
at 120, the default `thinking_medium` is 180, and `high/front_loaded` is 300. Every cell below
asks for `low/flat` and leaves `nsteps` alone, which is the cheapest setting the model was
trained for. Do not shorten the trajectory to save time instead: at `nsteps=20` with
`recycle_strategy=None` the run is far out of distribution and comes back as poly-alanine —
runnable, and scientifically meaningless.


## Unconditional generation

RFD4 always consumes an input structure — there is no size-only entry point — so "generate a
protein from nothing" has to be expressed as an input that describes nothing but a length.
Upstream's unconditional benchmark input is exactly that, and it ships here as
`unconditional_monomer_100.cif`: a single sentinel atom whose `res_name` is `100-100`, a collapsed
expandable segment meaning "between 100 and 100 residues", with NaN coordinates and every actual
condition value left unset. The annotation columns themselves are present — that is exactly the
point: `annotation_expsegmin/max_1_residue 100` is where the requested length lives, next to a
masked `condition_sequence_1_residue`, an indexed `condition_index_1_residue`, and a NaN
`_condition_coordinate_1_atom` entry.

That file only means anything if its annotations are read, which is what
`input_as_annotated_cif=True` asks for: the CIF is handed to RFD4 byte-for-byte so its
`condition_*` and `annotation_*` columns survive, instead of being treated as a raw structure to
clean up and re-annotate. No `contig` is needed — the sentinel already carries the topology — and
the result is a 100-residue monomer whose sequence and all-atom coordinates were generated
together. One replicate takes roughly 40 seconds on an H200.


In [6]:
unconditional_inputs = RFD4Input(
    design_specs=[
        RFD4DesignSpec(
            input_structure=monomer_sentinel,
            input_as_annotated_cif=True,
        )
    ]
)

# Cheapest in-distribution preset; nsteps stays at the 60 every preset pins it to
design_config = RFD4DesignConfig(
    num_replicates=1,
    batch_size=1,
    recycle_strategy="low/flat",
    seed=42,
    device="cuda",  # A GPU is required; RFD4 has no CPU path
)

unconditional_result = run_rfd4_design(unconditional_inputs, design_config)
print(unconditional_result)

Running run_rfd4_design [00:00]

RFD4Output(designed_structures=[1 bundles, 1 designs])


> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.


In [7]:
# Output is one bundle per input spec, each holding that spec's `num_replicates` designs
bundle = unconditional_result[0]
design = bundle[0]
sequence = design.complex.extract_protein_chains()[0][0]

print(f"Spec key: {bundle.spec_key}, designs in bundle: {len(bundle)}")
print(f"Chains: {len(design.complex.chains)}")
print(f"Designed sequence ({len(sequence)} aa): {sequence}")
print(f"Sentinel asked for 100 residues: {len(sequence) == 100}")
print(f"Metadata keys: {sorted(design.metadata)[:8]}")

design.structure.visualize()

Spec key: spec-0, designs in bundle: 1
Chains: 1
Designed sequence (100 aa): TVEKMEILLSLLEDDRVDVELALTLVENVEHLTVEKLEIILSLVEKGQVELALTLVENVEHLTVEKMEIILSLVEKGQVELALTLAEHVELTVEKMEIIL
Sentinel asked for 100 residues: True
Metadata keys: ['amino_acid_composition.ala_fraction', 'amino_acid_composition.gly_fraction', 'amino_acid_composition.n_generated_unknown_residues', 'amino_acid_composition.n_standard_residues', 'amino_acid_composition.val_fraction', 'chain_of_interest', 'checkpoint', 'checkpoint_revision']


## Binder design against PD-L1

De novo binder design is the canonical multi-chain RFD4 task, and the spec below is upstream's own
worked example — `docs/how-to/input_prep/examples/condition_spec_ppi.yaml`, the spec behind
`binder-in-10-lines.md` — transcribed into the typed fields and run against the same target:
PD-L1 (PDB [5O45](https://www.rcsb.org/structure/5O45)) chain A, residues 17-131.

Read the contig left to right: a de novo binder chain of 50-120 residues, a chain break, then the
whole target. The conditions then say what is known about that target.

- `rigid_groups` puts the target backbone in one internal-coordinate group, so the model knows its
  *shape* but not its *pose* — relative geometry within a group is known, global placement is not.
  That is what leaves the dock free to be designed. Use `fixed_coordinates` instead when the target
  should stay exactly where the input put it.
- `and not has_nan_coord()` drops atoms that have no coordinates to condition on. This cropped
  crystal structure carries 15 of them, across 6 unresolved sidechains, which is why upstream's own
  selection filters them rather than feeding NaNs into the group.
- `known_sequence` reveals the target's residue identities, which a real target always has. It
  doubles as a round-trip check: chain `A` comes back as the exact input PD-L1 sequence.
- `hotspots` names the target-interface atoms the binder should engage. The condition was trained on
  target atoms within 4.5 Å of any binder atom, so select specific sidechain atoms rather than whole
  residues. TYR56, MET115, and TYR123 are PD-L1's hotspot residues, and `sel('A/*/56/[CG,OH]')` is
  the CG and OH of residue 56.

The cell first compiles the spec with `write_spec_file()` — a cheap way to see what a set of typed
conditions turned into (`C_CRDGRP`, `C_SEQ`, `C_HOT`, …) before paying for a run — and then
generates. One replicate takes roughly 40 seconds on an H200.


In [8]:
binder_inputs = RFD4Input(
    design_specs=[
        RFD4DesignSpec(
            input_structure=pdl1_target,
            contig="50-120, /0, A17-131",
            rigid_groups={1: ["sel('A/*/*/[N,CA,C,O]') and not has_nan_coord()"]},
            known_sequence=["sel('A')"],
            hotspots=["sel('A/*/56/[CG,OH]')", "sel('A/*/115/[CG,SD]')", "sel('A/*/123/[CD2,OH]')"],
        )
    ]
)

# Compile the typed conditions into the upstream spec file, without generating anything
with tempfile.TemporaryDirectory() as temp_dir:
    spec_path, spec_keys = binder_inputs.write_spec_file(Path(temp_dir))
    print(f"Compiled {spec_path.name} for specs {spec_keys}:\n")
    print(spec_path.read_text())

binder_result = run_rfd4_design(binder_inputs, design_config)

native_target = pdl1_target.get_chain_sequence("A")
binder_design = binder_result[0][0]

# chain_sequence_map() is ligand-safe: a Fragment contributes its SMILES, not a sequence
for chain_id, sequence in binder_design.complex.chain_sequence_map().items():
    role = "target, revealed by known_sequence" if sequence == native_target else "de novo binder"
    print(f"chain {chain_id} ({len(sequence)} aa, {role})\n  {sequence}")

round_tripped = native_target in binder_design.complex.chain_sequence_map().values()
print(f"Target sequence round-tripped exactly: {round_tripped}")

binder_design.structure.visualize()

Compiled rfd4_input.yaml for specs ['spec-0']:

spec-0:
  input: /tmp/tmpytnpcfku/spec-0_input.cif
  contig: 50-120, /0, A17-131
  conditions:
    C_SEQ:
      true:
      - select: sel('A')
    C_CRDGRP:
      1:
      - select: sel('A/*/*/[N,CA,C,O]') and not has_nan_coord()
    C_HOT:
      true:
      - select: sel('A/*/56/[CG,OH]')
      - select: sel('A/*/115/[CG,SD]')
      - select: sel('A/*/123/[CD2,OH]')



Running run_rfd4_design [00:00]

chain B (56 aa, de novo binder)
  SVIDEVAEKYQLNSQEKNVLQAWVESDSIDPDVLGQFASSRQTVLLRAYIEVKDRL
chain A (115 aa, target, revealed by known_sequence)
  NAFTVTVPKDLYVVEYGSNMTIECKFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVN
Target sequence round-tripped exactly: True


### Symmetric oligomers are not supported

RFD4 documents symmetric oligomer design as *Planned* upstream, so there is no symmetry field on
`RFD4DesignSpec` and naming `symmetry` in `raw_conditions` is rejected with a pointer rather than
silently ignored. Use **`rfdiffusion3-design`** for symmetric homo-oligomer design; it implements
symmetric generation directly.


## Motif scaffolding

Motif scaffolding keeps a fragment that already does something useful — a binding loop, a
catalytic triad, an epitope — and asks the model to invent a protein that holds it in place. The
contig and the conditions play different roles here: the preserved contig segment decides *which*
residues enter the design and *where* in the chain they sit, while `fixed_coordinates` is what
actually hands the model their geometry. Pairing the selection with `is_protein_backbone` reveals
the motif backbone without revealing its sidechains, so they are repacked; adding
`known_sequence` reveals the identities too, which is what you want when the fragment's chemistry
is the point.

Below, residues 28-40 of the 65-residue fixture stand in for a motif. They are flanked by two
*named* expandable segments whose lengths the model chooses, and naming them means conditions can
address them — here the N-terminal scaffold segment is asked to be helical. For a motif whose
position in the chain should also be free, use `unindexed_groups` instead: selections sharing a
label keep their relative geometry while the model decides where they land.


In [9]:
motif_inputs = RFD4Input(
    design_specs=[
        RFD4DesignSpec(
            input_structure=fixture,
            contig="E1:8-16, A28-40, E2:8-16",
            fixed_coordinates=["sel('A/*/[28-40]') and is_protein_backbone"],
            known_sequence=["sel('A/*/[28-40]')"],
            secondary_structure={"HELIX": ["sel('*/E1')"]},
        )
    ]
)

motif_result = run_rfd4_design(motif_inputs, design_config)

motif_design = motif_result[0][0]
motif_sequence = motif_design.complex.extract_protein_chains()[0][0]
native_motif = fixture.get_chain_sequence("A")[27:40]

print(f"Scaffolded design: {len(motif_sequence)} residues (motif 13 + 16-32 scaffold)")
print(motif_sequence)
print(f"Native motif {native_motif} at index {motif_sequence.find(native_motif)} (-1 = not verbatim)")

Running run_rfd4_design [00:00]

Scaffolded design: 38 residues (motif 13 + 16-32 scaffold)
TELLEEAADKLIEEVLKRGLPVTVELDGKTYTIVPVKL
Native motif KLIEEVLKRGLPV at index 9 (-1 = not verbatim)


## Inverse folding

Inverse folding asks the opposite question from design: given a backbone, which sequences fold
into it? `run_rfd4_inverse_folding` holds the input backbone on independently resampled forward
marginals while resampling only the sequence and sidechain latents, so each replicate is an
independent sequence for the same fold — returned with its sidechains already packed, which the
message-passing alternatives (`proteinmpnn-sample`, `ligandmpnn-sample`, `fampnn-sample`) need a
separate packing step for.

Diversity is a temperature knob. `sequence_sampling.tsr_k` is a score-rescaling sharpness where
*higher* means *colder*, and the checkpoint default for the latent modality is `4.0`; dropping it
to `2.0` below raises the temperature and spreads the four sequences further apart. Conditions
still apply, so a run can hold part of the sequence with `known_sequence` or exclude an identity
outright with a `sequence_probabilities` entry setting `CYS` to `0.0`.

> **Inverse folding needs a complete all-atom backbone.** The 65-residue fixture is one, and
> recovers 29-35% of its native sequence at `tsr_k=2.0`. The cropped PD-L1 target above is *not*
> one: its 15 NaN-coordinate atoms across 6 unresolved crystallographic sidechains make it an
> incomplete structure, and pointed at it the model returns a CA-only poly-alanine trace. That is
> upstream behavior, not a implementation artifact — upstream's own CLI returns the identical degenerate
> result on that file in `generation=inverse_folding` mode. Repair or rebuild missing sidechains
> before inverse folding a crystal structure.


In [10]:
# No contig: the whole input backbone is kept and only its sequence is redesigned
folding_inputs = RFD4Input(design_specs=[RFD4DesignSpec(input_structure=fixture)])

folding_config = RFD4InverseFoldingConfig(
    num_replicates=4,
    batch_size=1,
    recycle_strategy="low/flat",
    sequence_sampling={"tsr_k": 2.0},  # Hotter than the 4.0 default, for more diverse sequences
    seed=42,
    device="cuda",  # A GPU is required; RFD4 has no CPU path
)

folding_result = run_rfd4_inverse_folding(folding_inputs, folding_config)

native = fixture.get_chain_sequence("A")
print(f"native  {native}")
for index, folded in enumerate(folding_result[0]):
    designed = folded.complex.extract_protein_chains()[0][0]
    identical = sum(a == b for a, b in zip(designed, native, strict=False))
    print(f"seq {index}   {designed}  ({identical / len(native):.0%} native recovery)")

Running run_rfd4_inverse_folding [00:00]

native  MRKKLDLKKFVEDKNQEYAARALGLSQKLIEEVLKRGLPVYVETNKDGNIKVYITQDGITQPFPP
seq 0   TWSLLDLKEYVEEYGMREAARELEVTERSLYHWLENGYSVRVRTNEDGSLDVEVKVNDTWTLVGT  (29% native recovery)
seq 1   ALETVDLVEFVNEHGINAAAEHLGVSKTTLYHWMEKGYTVKVYTEDDGTLKVYVQVDGVSSLHGA  (31% native recovery)
seq 2   ALEKLDLTEFIAEKGYTEAARTLNVSRSTLHTWKEHEYPVYVERAEDGALWVYVEVDGELHRVGV  (34% native recovery)
seq 3   SLTELDLHELVEEMGLDKAAELLGVSKDDLKEWMANSLAVYVETMSDGTLLVYVEKDGELNLVNA  (35% native recovery)


## Export results

Designs export to mmCIF (the default) or PDB, one file per design named
`{spec_key}_design_{index}.{format}` inside the directory you name. mmCIF preserves the
all-atom detail and metadata RFD4 generates; PDB is the more portable format for PyMOL and
ChimeraX. Validate designs by refolding the exported sequences with a structure predictor and
comparing back to the generated backbone — RFD4 attaches no confidence score of its own.


In [11]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

binder_result.export("rfd4_binders_cif", export_path=output_dir, file_format="cif")
print(f"CIF binder designs exported to {output_dir / 'rfd4_binders_cif/'}")

binder_result.export("rfd4_binders_pdb", export_path=output_dir, file_format="pdb")
print(f"PDB binder designs exported to {output_dir / 'rfd4_binders_pdb/'}")

folding_result.export("rfd4_inverse_folding_cif", export_path=output_dir, file_format="cif")
print(f"Inverse-folding designs exported to {output_dir / 'rfd4_inverse_folding_cif/'}")

CIF binder designs exported to example_output/rfd4_binders_cif
PDB binder designs exported to example_output/rfd4_binders_pdb
Inverse-folding designs exported to example_output/rfd4_inverse_folding_cif
